In [54]:
import numpy as np
experimental_set=np.load('experimental_set_R1.npy')

In [55]:
# Define simulation function
def simulate_loss(mu, sigma, weights, size, shape_demand, scale_demand):
    samples_array = np.array([np.random.lognormal(mean=np.log(mu[i]), sigma=sigma[i], size=size) for i in range(len(mu))])
    # gamma distribution
    # shape=2.0，scale=2.0
    demand = np.random.gamma(shape=shape_demand, scale=scale_demand, size=size)
    remaining_demand = np.maximum(demand - np.sum(weights), 0)
    purchase_amounts = np.vstack([
    np.full(size, weights[0]),
    np.full(size, weights[1]),
    remaining_demand])
    loss = np.sum(purchase_amounts * samples_array, axis=0)
    # weights_price_impact = np.exp(weights)-1
    # weights_price_impact=weights
    # weighted_samples = np.dot(weights_price_impact, samples_array)
    return loss


# Define CVaR calculation
def calculate_cvar_weighted(mu, sigma, weights, alpha, shape_demand, scale_demand):
    size = 10**5  # Reduced size for efficiency
    samples = simulate_loss(mu, sigma, weights, size, shape_demand, scale_demand)
    sorted_samples = np.sort(samples)  # Sort samples directly
    var_alpha = np.percentile(sorted_samples, (alpha) * 100)#0.95 quantile
    subtracted_samples=samples-var_alpha
    postive_part=np.where(subtracted_samples > 0, subtracted_samples, 0)
    cvar_alpha=(np.mean(postive_part)/(1-alpha))+var_alpha  
    cvar_variance=np.var(postive_part, ddof=1)/((1-alpha)**2)
    expected = np.mean(samples)
    variance = np.var(samples, ddof=1)
    return expected, cvar_alpha, variance,cvar_variance

In [56]:
import scipy.stats as stats
# Define parameters
mu = np.array([0.5, 1, 2])  # Mean list
sigma = np.array([2, 1.5, 1])      # Standard deviation list
#mu = 1 + mu
sigma = np.sqrt(sigma)

tail_quantile_demand=0.995
shape_demand=2.0
scale_demand=2.0
percentail_demand=stats.gamma.ppf(tail_quantile_demand, shape_demand, scale_demand)
w1_values = np.linspace(0, percentail_demand, 101)
w2_values = np.linspace(0, percentail_demand, 101)

# Generate feasible region
W1, W2 = np.meshgrid(w1_values, w2_values)
W3 = 1 - W1 - W2
weights = np.vstack([W1.ravel(), W2.ravel()]).T
feasible_region = weights[(weights >= 0).all(axis=1) & (weights.sum(axis=1) <=percentail_demand)]
risk_quantile=0.95

from scipy import stats

sample_size=10**6


num_soultions=len(experimental_set)
LB_list=np.empty((num_soultions,2))
UB_list=np.empty((num_soultions,2))

#Confidence level for the confidence regions Denoted as \alpha in the paper
overall_confidence=0.95


confidence_per_solution=1-(overall_confidence)**(1/num_soultions)
num_objectives=2
confidence_per_dimension=1-(confidence_per_solution)/num_objectives

for rep in range(num_soultions):
    print(rep)
    weight = experimental_set[rep]
    expected_loss, cvar_loss,variance,cvar_variance = calculate_cvar_weighted(mu, sigma, weight, risk_quantile,shape_demand,scale_demand)
    print(expected_loss, cvar_loss)
    expected_loss_HW=np.sqrt(variance/sample_size)
    cvar_loss_HW=np.sqrt(cvar_variance/sample_size)
    norm_critical = stats.t.ppf((1 + confidence_per_dimension) / 2,sample_size-1)
    LB_list[rep][0]=expected_loss-norm_critical*expected_loss_HW
    UB_list[rep][0]=expected_loss+norm_critical*expected_loss_HW
    norm_critical = stats.t.ppf((1 + confidence_per_dimension) / 2, sample_size-1)
    LB_list[rep][1]=cvar_loss-norm_critical*cvar_loss_HW
    UB_list[rep][1]=cvar_loss+norm_critical*cvar_loss_HW


0
17.899945656557993 103.2927494015059
1
14.221810203489445 80.59363170144192
2
10.399061823122699 61.86892062840049
3
11.0896074674365 67.03300603712238
4
18.101109684261136 114.92010636325546
5
13.303634204177023 97.01839869671261
6
12.669382973680843 69.35034557279415
7
14.823455775507442 82.88619649505182
8
15.112733323139949 86.65025270534471
9
14.186824274192702 88.40881063186158
10
9.713404841029675 59.824352205166605
11
10.946624163651563 61.25712401221223
12
13.612525504825442 83.06958763505105
13
11.921345982264844 71.75177949222416
14
12.010935986907011 73.26623052857639
15
9.405883915984127 63.496215436094204
16
16.126652378592073 103.1159825809849
17
9.941584466958473 71.44109642019654
18
11.699192359212013 67.72851280256494
19
12.792048327014582 74.72289430744965
20
17.544205719529035 103.5047288456231
21
12.254425447646444 81.9352090315586
22
10.868741998218983 61.80626225625337
23
11.156897971340632 64.22002989716196
24
15.560046510236752 85.9215461999845
25
15.42090635

In [57]:
np.save('LB_list_R1.npy',LB_list)
np.save('UB_list_R1.npy',UB_list)

